In [ ]:
import math

In [ ]:
# 1. Value nesnesi: önce sadece sayıyı saklayan en basit hâli.
class Value():

  def __init__(self,data):

    self.data = data

  def __repr__(self):
    return f"Value (data = {self.data})"

a = Value(2.0)
a

In [ ]:
class Value():

  def __init__(self,data, children = (),_op = '',label = ''):

    self.data = data
    self._prev = set(children)
    self._op = _op
    self.grad = 0.0
    self.label = label
    self._backward = lambda: None

  def __repr__(self):
    return f"Value (data = {self.data})"

  def __add__(self,other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self,other), "+")
    def _backward():
      self.grad += out.grad * 1.0
      other.grad += out.grad * 1.0
    out._backward = _backward
    return out
  def __pow__(self,other):
    assert isinstance(other, (int,float))
    out = Value(self.data**other,(self,), f'**{other}')
    def _backward():
      self.grad += (other * self.data**(other - 1) * out.grad)
    out._backward = _backward
    return out
  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self,other), "*")
    def _backward():
      self.grad += out.grad * other.data
      other.grad += out.grad * self.data
    out._backward = _backward
    return out
  def __rmul__(self,other):
    return self * other
  def __radd__(self,other):
    return self + other
  def __truediv__(self,other):
    return self * other ** -1
  def __neg__(self):
    return self * -1
  def __sub__(self,other):
    return self + (-other)
  def tanh(self):
    e = math.e
    x = self.data
    y = (e**(2*x) - 1) / (e**(2*x) + 1)
    output = Value(y, children=(self,), _op = 'tanh')
    def _backward():
      self.grad += output.grad * (1- y**2)
    output._backward =  _backward
    return output

  def exp(self):
     x = self.data
     out = Value(math.exp(x), (self,), 'exp')
     def _backward():
      self.grad += out.data * out.grad
     out._backward =  _backward
     return out
  # 3. Çıkıştan başlayıp düğümleri ters sırada geziyorum.
  def backward(self):
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)
    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

a = Value(5.0, label = "a")
b = Value(6.0, label = "b")
c = Value(7.0, label = "c")
d = a * b
d.label = 'd'
e = c + d
e.label = 'e'
f = Value(-5.0, label = 'f')
L = e * f
L.label = "L"
L

In [ ]:
x = Value(3.0)
y = x**2
y.backward()
print(y.data,x.grad)

In [ ]:
a = Value(2)
b = Value(4)
c = a / b
c.backward()
print(c.data,a.grad,b.grad)

In [ ]:
x = Value(2)
y = x.exp()
y.backward()
print(x.grad,y)

In [ ]:
print(a + 1)
print(2 * a)
print(a * 2)

In [ ]:
try:
  from graphviz import Digraph
except ModuleNotFoundError:
  Digraph = None
  print('Graphviz yüklü değil; grafik hücreleri atlanacak.')

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  if Digraph is None:
    return None

  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right

  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
    if n._op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._op, label = n._op)
      # and connect this node to it
      dot.edge(uid + n._op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)

  return dot

In [ ]:
draw_dot(L)

## 2. Önce elle gradient hesaplama

`backward()` yazmadan önce küçük grafikte zincir kuralını elle uyguluyorum. Böylece her sayının nereden geldiği daha görünür oluyor.

In [ ]:
def numerical_derivative_a():

  h = 0.0001

  a = Value(5.0, label = "a")
  b = Value(6.0, label = "b")
  c = Value(7.0, label = "c")
  d = a * b
  d.label = 'd'
  e = c + d
  e.label = 'e'
  f = Value(-5.0, label = 'f')
  L = e * f
  L.label = "L"
  L1 = L.data

  a = Value(5.0, label = "a")
  a.data += h
  b = Value(6.0, label = "b")
  c = Value(7.0, label = "c")
  d = a * b
  d.label = 'd'
  e = c + d
  e.label = 'e'
  f = Value(-5.0, label = 'f')
  L = e * f
  L.label = "L"
  L2 = L.data

  print('Sayısal türev (dL/da):', (L2 - L1) / h)

numerical_derivative_a()

∂L/∂c = ∂L/∂e × ∂e/∂c
∂L/∂d = ∂L/∂e × ∂e/∂d

In [ ]:
L.grad = 1
e.grad = float(f.data) * 1.0 # L.grad =  1 olduğu için
f.grad = float(e.data) * 1.0 # L.grad =  1 olduğu için

∂e/∂c = 1
∂e/∂d = 1

In [ ]:
c.grad = e.grad * 1.0 # toplama nın türevi olduğu için  1.0
d.grad = e.grad * 1.0 # toplama nın türevi olduğu için  1.0

In [ ]:
a.grad = d.grad * b.data
b.grad = d.grad * a.data

In [ ]:
w1 = Value(2.0,label = "weight 1")
w2 = Value(0.8,label = "weight 2")
x1 = Value(-3.0,label = "input 1")
x2 = Value(1.0, label="input 2")
b = Value(6.8813735870195432, label = "bias")


w1x1 = w1 * x1
w1x1.label = 'w1*x1'

w2x2 = w2 * x2
w2x2.label = 'w2*x2'
result = w1x1 + w2x2
result.label = "result"
n = result + b
n.label = "n"
o = n.tanh()  # tanh, Value sınıfının bir metodu olduğu için n üzerinden çağrılıyor.
o.label = 'output'
# o.grad = 1
# n.grad = o.grad * (1 - o.data **2)
# result.grad = n.grad * 1
# b.grad = n.grad * 1
# w1x1.grad = result.grad * 1
# w2x2.grad = result.grad * 1
# w1.grad = w1x1.grad * x1.data
# x1.grad = w1x1.grad * w1.data
# w2.grad = w2x2.grad * x2.data
# x2.grad = w2x2.grad * w2.data


In [ ]:
# Bu hücrede gradyanları bilerek elle yazıyorum.
o.grad = 1.0
n.grad = o.grad * (1 - o.data**2)
result.grad = n.grad * 1.0
b.grad = n.grad * 1.0
w1x1.grad = result.grad * 1.0
w2x2.grad = result.grad * 1.0
w1.grad = w1x1.grad * x1.data
x1.grad = w1x1.grad * w1.data
w2.grad = w2x2.grad * x2.data
x2.grad = w2x2.grad * w2.data

print('output:', o.data)
print('w1, x1, w2, x2, b:', w1.grad, x1.grad, w2.grad, x2.grad, b.grad)

In [ ]:
# Aynı değişken iki kez kullanılırsa grad değerleri toplanmalı.
a = Value(3.0, label='a')
w3 = a + a
w3.label = 'a + a'
w3.backward()
print('a.grad =', a.grad)  # 2.0 olmalı


In [ ]:
# Aynı neuron'u bu kez otomatik backward() ile çalıştırıyorum.
w1 = Value(2.0, label='weight 1')
w2 = Value(0.8, label='weight 2')
x1 = Value(-3.0, label='input 1')
x2 = Value(1.0, label='input 2')
b = Value(6.8813735870195432, label='bias')

w1x1 = w1 * x1
w1x1.label = 'w1*x1'
w2x2 = w2 * x2
w2x2.label = 'w2*x2'
result = w1x1 + w2x2
result.label = 'result'
n = result + b
n.label = 'n'
o = n.tanh()
o.label = 'output'

o.backward()
print('output:', o.data)
print('w1, x1, w2, x2, b:', w1.grad, x1.grad, w2.grad, x2.grad, b.grad)

In [ ]:
draw_dot(o)

In [ ]:
w1 = Value(2.0, label='weight 1')
w2 = Value(0.8, label='weight 2')
x1 = Value(-3.0, label='input 1')
x2 = Value(1.0, label='input 2')
b = Value(6.8813735870195432, label='bias')

w1x1 = w1 * x1
w1x1.label = 'w1*x1'

w2x2 = w2 * x2
w2x2.label = 'w2*x2'

result = w1x1 + w2x2
result.label = 'result'

n = result + b
n.label = 'n'

# tanh(x) = (e^(2x) - 1) / (e^(2x) + 1)
e2n = (2 * n).exp()
e2n.label = 'e2n'
o = (e2n - 1) / (e2n + 1)
o.label = 'output'
print('output:', o.data)

In [ ]:
o.backward()
backward_grads = {
    'w1': w1.grad, 'x1': x1.grad, 'w2': w2.grad,
    'x2': x2.grad, 'b': b.grad,
}
print('Parçalanmış tanh output:', o.data)
print('Parçalanmış tanh gradyanları:', backward_grads)
draw_dot(o)


### PyTorch ile kontrol

Aynı işlemi şimdi PyTorch'a yaptırıyorum. `requires_grad=True`, o değişkenin türevini takip etmesini açıyor. `float64` kullanmamın sebebi de küçük farkları daha net görmek.

Sonuç ve `.grad` değerleri kendi `Value` sınıfımdan çıkan değerlerle aynı olmalı.

In [ ]:
try:
    import torch
except ModuleNotFoundError:
    torch = None
    print('PyTorch yüklü değil; bu kontrolü Colab veya PyTorch yüklü ortamda çalıştırmalıyım.')

if torch is not None:
    x1 = torch.tensor([-3.0], dtype=torch.float64, requires_grad=True)
    x2 = torch.tensor([1.0], dtype=torch.float64, requires_grad=True)
    w1 = torch.tensor([2.0], dtype=torch.float64, requires_grad=True)
    w2 = torch.tensor([0.8], dtype=torch.float64, requires_grad=True)
    b = torch.tensor([6.8813735870195432], dtype=torch.float64, requires_grad=True)

    n = x1 * w1 + x2 * w2 + b
    o = torch.tanh(n)
    o.backward()

    print('output:', o.item())
    print('w1, x1, w2, x2, b:', w1.grad.item(), x1.grad.item(), w2.grad.item(), x2.grad.item(), b.grad.item())


In [ ]:
# Sayısal türev kontrolünü aşağıda, forward_pass tanımlandıktan sonra yapıyorum.

### Sayısal türev ile kontrol

Burada `Value` kullanmıyorum. Bir parametreyi çok küçük miktarda değiştirip output farkına bakıyorum. Bu yaklaşık türev, `backward()` sonucuna yakın çıkmalı.

In [ ]:
def forward_pass(w1,x1,w2,x2,b):

  n = (w1 * x1) + (w2 * x2) + b
  e2n = math.exp(2*n)
  output = (e2n - 1) / (e2n + 1)
  return output
w1 = 2.0
w2 = 0.8
x1 = -3.0
x2 = 1.0
b = 6.8813735870195432

print(forward_pass(w1, x1, w2, x2, b))

In [ ]:
h = 0.000001

v1 = forward_pass(w1, x1, w2, x2, b)

w1_grad = (forward_pass(w1 + h, x1, w2, x2, b) - v1) / h
x1_grad = (forward_pass(w1, x1 + h, w2, x2, b) - v1) / h
w2_grad = (forward_pass(w1, x1, w2 + h, x2, b) - v1) / h
x2_grad = (forward_pass(w1, x1, w2, x2 + h, b) - v1) / h
b_grad = (forward_pass(w1, x1, w2, x2, b + h) - v1) / h

print(w1_grad)
print(x1_grad)
print(w2_grad)
print(x2_grad)
numerical_grads = {
    'w1': w1_grad, 'x1': x1_grad, 'w2': w2_grad,
    'x2': x2_grad, 'b': b_grad,
}
print(numerical_grads)

In [ ]:
if torch is not None:
    w1_t = torch.tensor([2.0], dtype=torch.float64, requires_grad=True)
    w2_t = torch.tensor([0.8], dtype=torch.float64, requires_grad=True)
    x1_t = torch.tensor([-3.0], dtype=torch.float64, requires_grad=True)
    x2_t = torch.tensor([1.0], dtype=torch.float64, requires_grad=True)
    b_t = torch.tensor([6.8813735870195432], dtype=torch.float64, requires_grad=True)

    n_t = w1_t * x1_t + w2_t * x2_t + b_t
    e2n_t = torch.exp(2 * n_t)
    o_t = (e2n_t - 1) / (e2n_t + 1)
    o_t.backward()

    torch_grads = {
        'w1': w1_t.grad.item(), 'x1': x1_t.grad.item(),
        'w2': w2_t.grad.item(), 'x2': x2_t.grad.item(),
        'b': b_t.grad.item(),
    }

    print('output:', o_t.item())
    print('\nKarşılaştırma:')
    for name in backward_grads:
        print(f"{name}: Value={backward_grads[name]:.10f}, sayısal={numerical_grads[name]:.10f}, PyTorch={torch_grads[name]:.10f}")
else:
    print('PyTorch kontrolü için bu hücreyi PyTorch yüklü ortamda çalıştırmalıyım.')

### Kısa sonuç

Kendi `backward()` hesabım, küçük `h` ile yaptığım sayısal türev ve PyTorch aynı gradyanlara geliyor. Küçük farklar varsa, bunlar sayısal türevin yaklaşık hesap yapmasından kaynaklanır.